# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
!git clone https://github.com/Ebrahim827/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

pd.set_option("display.width", 120)
RANDOM_SEED = 42

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
df["y_declining"] = (df["trend_direction"] == "down").astype(int)

numeric_features = ["days_since_last_update", "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "engaged_sessions_90d", "scroll_events_90d", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct", "word_count", "content_age_days",
    "search_volume", "competition", "cpc", "days_with_impressions", "days_with_sessions"]
categorical_features = ["content_type", "main_intent", "competition_level"]
numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                       ("ohe", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])
X = df[numeric_features + categorical_features]
y = df["y_declining"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    k = min(k, len(order))
    return np.asarray(labels)[order[:k]].mean()

print("Loaded", df.shape[0], "rows. Setup done.")

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 229, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 229 (delta 96), reused 59 (delta 59), pack-reused 111 (from 3)
Receiving objects: 100% (229/229), 1.89 MiB | 13.41 MiB/s, done.
Resolving deltas: 100% (111/111), done.
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Loaded 30000 rows. Setup done.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 — "What Predicts Health?"**
The paper trained a model (Random Forest) to guess a page's "Health Score" using different page stats. It found that two stats — Average Position (43%) and Impressions (32%) — mattered way more than anything else, together making up over 75% of what the model looked at.

**My question about it:** The paper's own Health Score formula is built directly from Impressions, Position, CTR, and Scroll Depth. So Average Position and Impressions aren't just related to the score — they're literally ingredients in it. That's a bit like training a model to predict "total price" and being impressed that it heavily uses "item cost" — of course it does, it's built from it. The paper does admit this a little, saying the result is "descriptive, not causal," which is good and honest. But my question is: how much of that 75% is the model just re-finding its own formula, versus real new information? A simple way to check: remove Position and Impressions from the model, retrain it, and see if it can still predict Health Score well using only the leftover stats. If it falls apart without them, that confirms most of the "insight" was circular.


**Finding 2 — "What Predicts Growth?"**
The paper trained a Logistic Regression model to guess whether a page is growing or declining, and got 71% accuracy, tested with an 80/20 split (80% of data to train, 20% held back to test).

**My question about it**: There are 341,701 pages but only 57 brands total — so each brand owns thousands of pages. If the 80/20 split was done randomly (just shuffling all rows), pages from the same brand would likely end up on both sides of the split — some in training, some in testing. That means the model could partly be learning "this looks like Brand X's writing style" instead of a real, general pattern that would work for a brand it's never seen before. That would make the 71% look better than it really is. The paper's methodology section doesn't say whether the split was grouped by brand or not. A simple way to check: redo the split so that each brand's pages are entirely in training OR entirely in testing (never split across both), then see if accuracy stays close to 71% or drops.

In [11]:
print("Section 1 done. See the two findings and questions written above.")

Section 1 done. See the two findings and questions written above.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

This is basically the same question I just asked about the paper (Finding 2), except now I'm checking my OWN model for the same problem.
I trained the exact same model twice, on the exact same data — once with a normal random split, once with a grouped split (each client entirely on one side).
Real result:

**Random split (not honest):** precision@20 = 1.00 (base rate 0.545)
**Grouped split (honest):** precision@20 = 0.55 (base rate 0.559)

The random split looked like a perfect model — but that was fake. The gap between **1.00 and 0.55** shows the model wasn't actually learning what makes a page decline; it was mostly just recognizing specific clients it had already seen in training and remembering how they behaved. Once clients are kept fully separate between training and testing (the honest way), the score crashes almost all the way down to the base rate — meaning the model barely does better than guessing on a client it's never seen before. This is a genuinely important finding: without the grouped split, this project would have reported a fake, inflated number.

In [12]:
# BEFORE: normal random split (does NOT keep each client together)
tr_random, te_random = train_test_split(np.arange(len(df)), test_size=0.3, random_state=RANDOM_SEED)

# AFTER: grouped split (keeps each client entirely on one side)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=RANDOM_SEED)
tr_grouped, te_grouped = next(gss.split(df, groups=df["client_id"]))

def train_and_score(train_idx, test_idx):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    rf = Pipeline([("prep", preprocess), ("clf", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, random_state=RANDOM_SEED))])
    rf.fit(X_train, y_train)
    proba = rf.predict_proba(X_test)[:, 1]
    return {
        "base_rate": round(y_test.mean(), 3),
        "precision@20": round(precision_at_k(proba, y_test.values, 20), 3),
    }

before_random = train_and_score(tr_random, te_random)
after_grouped = train_and_score(tr_grouped, te_grouped)

print("BEFORE — random split (not honest):", before_random)
print("AFTER  — grouped by client (honest):", after_grouped)

BEFORE — random split (not honest): {'base_rate': np.float64(0.545), 'precision@20': np.float64(1.0)}
AFTER  — grouped by client (honest): {'base_rate': np.float64(0.559), 'precision@20': np.float64(0.55)}


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Proof test result:** Without the banned feature (trend_pct), the model scores 0.55 at precision@20 — the honest number. The moment I add trend_pct back in on purpose, the score jumps to a perfect 1.00. This confirms trend_pct really would have been cheating if I'd used it — the model isn't learning a real pattern, it's just reading the label in disguise. This also proves my testing method works correctly: adding a known-bad feature should make the score jump toward 1.0, and it did.

In [13]:
leakage_audit = pd.DataFrame([
    {"feature": "days_since_last_update", "safe?": "yes", "why": "known before prediction, has nothing to do with the label"},
    {"feature": "impressions_90d, clicks_90d, sessions_90d, pageviews_90d", "safe?": "yes", "why": "past 90-day stats, not built from the label"},
    {"feature": "ctr, engagement_rate, scroll_rate", "safe?": "yes", "why": "built from past clicks/sessions, not from trend_direction"},
    {"feature": "avg_position", "safe?": "yes", "why": "search ranking at snapshot time, not a decision someone already made"},
    {"feature": "content_type, main_intent, competition_level", "safe?": "yes", "why": "fixed facts about the page, don't change over time"},
    {"feature": "content_age_days, word_count", "safe?": "yes", "why": "fixed facts about the page"},
    {"feature": "trend_direction, trend_pct", "safe?": "NOT USED", "why": "this is basically the answer key — the label is built from this, so it's banned as an input"},
    {"feature": "client_id, content_id", "safe?": "NOT USED as a feature", "why": "just ID codes — only used to keep clients together during the split, never fed to the model"},
])
print(leakage_audit.to_string(index=False))

# Proof test: add the banned feature on purpose and watch the score jump.
# If it doesn't jump, our testing method itself would be broken.
leaky_numeric = numeric_features + ["trend_pct"]
preprocess_leaky = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), leaky_numeric),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                       ("ohe", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])
X_leaky = df[leaky_numeric + categorical_features]
Xl_train, Xl_test = X_leaky.iloc[tr_grouped], X_leaky.iloc[te_grouped]
yl_train, yl_test = y.iloc[tr_grouped], y.iloc[te_grouped]

rf_leaky = Pipeline([("prep", preprocess_leaky), ("clf", RandomForestClassifier(
    n_estimators=300, min_samples_leaf=5, random_state=RANDOM_SEED))])
rf_leaky.fit(Xl_train, yl_train)
proba_leaky = rf_leaky.predict_proba(Xl_test)[:, 1]

print()
print("precision@20 WITHOUT the banned feature (honest):", after_grouped["precision@20"])
print("precision@20 WITH the banned feature added on purpose (should jump up):",
      round(precision_at_k(proba_leaky, yl_test.values, 20), 3))

                                                 feature                 safe?                                                                                         why
                                  days_since_last_update                   yes                                   known before prediction, has nothing to do with the label
impressions_90d, clicks_90d, sessions_90d, pageviews_90d                   yes                                                 past 90-day stats, not built from the label
                       ctr, engagement_rate, scroll_rate                   yes                                   built from past clicks/sessions, not from trend_direction
                                            avg_position                   yes                        search ranking at snapshot time, not a decision someone already made
            content_type, main_intent, competition_level                   yes                                          fixed facts about the pag

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Here's one of my boldest sentences from my Week-5 notebook, and a safer, more honest rewrite of it:

**Original (too bold):** "The model predicts which pages are declining."

**Rewritten (safe):** "The model's score is directionally associated with pages that were observed to be declining in this dataset. It's useful as decision-support for prioritizing which pages to look at — not a guarantee about what will happen to any single page."

**Why the rewrite is better:** the original sentence sounds like a fact/certainty ("predicts" = will definitely happen). The rewrite uses softer, more honest words — "observed," "associated," "decision-support" — that match what the model actually showed, without overselling it.

In [14]:
safe_words = ["observed", "measured", "directional", "decision-support", "associated"]
risky_words = ["predicts", "proves", "guarantees", "causes", "always", "will"]
print("Words I used in my rewrite (safe):", safe_words)
print("Words I avoided (too strong/risky):", risky_words)

Words I used in my rewrite (safe): ['observed', 'measured', 'directional', 'decision-support', 'associated']
Words I avoided (too strong/risky): ['predicts', 'proves', 'guarantees', 'causes', 'always', 'will']


## Self-check

Before you submit, confirm each line honestly:

- [✔ ] Every section above is filled — markdown thinking AND the code that backs it
- [✔ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔ ] No client names, URLs, or private queries anywhere
- [✔ ] My claims use careful words: observed, measured, directional, decision-support
- [✔ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.